In [35]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity





In [26]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

print(movies.shape)
print(credits.shape)

(4803, 20)
(4803, 4)


In [3]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [4]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [28]:
# Rename column for merging
credits.columns = ["id", "title", "cast", "crew"]

# Merge
movies = movies.merge(credits, on="title")

print(movies.shape)
movies.head(1)

(4821, 26)


,budget,genres,homepage,id_x,keywords,original_language,original_title,overview,popularity,production_companies,...,tagline,title,vote_average,vote_count,id_y,cast_x,crew_x,id,cast_y,crew_y
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [10]:
movies.columns.tolist()

['budget',
 'genres',
 'homepage',
 'id_x',
 'keywords',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'spoken_languages',
 'status',
 'tagline',
 'title',
 'vote_average',
 'vote_count',
 'id_y',
 'cast_x',
 'crew_x',
 'id',
 'cast_y',
 'crew_y']

In [29]:
movies = movies[["id_x", "title", "overview", 
                 "genres", "keywords", "cast_x", "crew_x"]]

# Rename for clarity
movies.columns = ["movie_id", "title", "overview", 
                  "genres", "keywords", "cast", "crew"]

print(movies.shape)
movies.head()

(4821, 7)


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [30]:
import ast
def convert (text):
    result=[]
    for item in ast.literal_eval(text):
        result.append(item["name"])
    return result
def convert_cast(text):
    result=[]
    for i,item in enumerate(ast.literal_eval(text)):
        if i==3:
            break
        result.append(item["name"])
    return result
    
def fetch_director(text):
    for item in ast.literal_eval(text):
        if item["job"]=="Director":
            return[item["name"]]
    return[]


movies["genres"]=movies["genres"].apply(convert)
movies["keywords"]=movies["keywords"].apply(convert)
movies["cast"]=movies["cast"].apply(convert_cast)
movies["crew"]=movies["crew"].apply(fetch_director)
movies[["genres","keywords"]].head()

,genres,keywords
0,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon..."
1,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ..."
2,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi..."
3,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i..."
4,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel..."


In [31]:
# remove spaces from the text
movies["genres"]   = movies["genres"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["keywords"] = movies["keywords"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["cast"]     = movies["cast"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["crew"]     = movies["crew"].apply(lambda x: [i.replace(" ", "") for i in x])
# convert overview from a sentence -- to list of words
movies["overview"]=movies["overview"].apply(lambda x:x.split() if isinstance(x,str)else[])

In [32]:
# combine all columns into one column tags
movies["tags"]=(movies["genres"]+movies["keywords"]+movies["cast"]+movies["crew"]+movies["overview"])
#join the list into single lowercase string
movies["tags"]=movies["tags"].apply(lambda x:" ".join(x).lower())
# keep only what we want
movies=movies[["movie_id","title","tags"]]
movies.head()



,movie_id,title,tags
0,19995,Avatar,action adventure fantasy sciencefiction cultur...
1,285,Pirates of the Caribbean: At World's End,adventure fantasy action ocean drugabuse exoti...
2,206647,Spectre,action adventure crime spy basedonnovel secret...
3,49026,The Dark Knight Rises,action crime drama thriller dccomics crimefigh...
4,49529,John Carter,action adventure sciencefiction basedonnovel m...


## Vectorizer

In [34]:
cv=CountVectorizer(max_features=5000,stop_words="english")
vectors=cv.fit_transform(movies['tags']).toarray()
print(vectors.shape)



(4821, 5000)


### Cosine Similarity

In [36]:
similarity=cosine_similarity(vectors)
print(similarity.shape)

(4821, 4821)


### Recommendation function

In [43]:

def recommend(movie):
    idx = movies[movies["title"] == movie].index[0]
    distance = sorted(list(enumerate(similarity[idx])),
                     reverse=True,
                     key=lambda x: x[1])
    # these lines should be INSIDE the function
    print(f"\n🎬 Because you watched '{movie}', we recommend:\n")
    for i in distance[1:6]:
        print(movies.iloc[i[0]].title)

recommend("Batman Begins")


🎬 Because you watched 'Batman Begins', we recommend:

The Dark Knight
The Dark Knight Rises
Batman
Batman
Batman & Robin


### Save the model

In [44]:
import pickle
pickle.dump(movies,open("movies.pkl","wb"))
pickle.dump(similarity,open("similarity.pkl","wb"))
print("model saved successfully!")

model saved successfully!
